In [1]:
import os
import io
import zipfile
import requests
import frontmatter
import re
import json
import secrets
import random
import asyncio
from pathlib import Path
from datetime import datetime
from typing import List, Any
from tqdm.auto import tqdm
import numpy as np
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer
from pydantic_ai import Agent
from openai import AsyncOpenAI
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.models.openai import OpenAIChatModel
import openai
from pydantic import BaseModel

os.environ["OPENROUTER_API_KEY"] = "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"  


c:\Users\rabi3\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def ingest_fastapi_docs():
    url = 'https://codeload.github.com/fastapi/fastapi/zip/refs/heads/master'
    resp = requests.get(url)
    if resp.status_code != 200:
        raise Exception("Failed to download repository")

    repo_data = []
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for file_info in zf.infolist():
            filename = file_info.filename
            if 'docs/en/docs/' in filename and filename.endswith('.md'):
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode('utf-8', errors='ignore')
                    post = frontmatter.loads(content)
                    data = post.to_dict()          
                    data['filename'] = filename
                    repo_data.append(data)
    return repo_data

fastapi_data = ingest_fastapi_docs()
print(f"Successfully ingested {len(fastapi_data)} documentation files!")
if fastapi_data:
    print("Sample keys:", list(fastapi_data[0].keys()))


Successfully ingested 153 documentation files!
Sample keys: ['content', 'filename']


In [3]:
def split_markdown_by_level(text, level=2):
    """Split by ## (or any header level) – preserves logical sections"""
    header_pattern = r'^(\#{1,' + str(level) + r'} .+)$'
    parts = re.split(header_pattern, text, flags=re.MULTILINE)
    sections = []
    for i in range(1, len(parts), 2):
        if i + 1 < len(parts):
            header = parts[i].strip()
            content = parts[i + 1].strip()
            if content:
                sections.append(f"{header}\n\n{content}")
    return sections

fastapi_chunks = []
for doc in fastapi_data:
    sections = split_markdown_by_level(doc.get('content', ''), level=2)
    doc_title = doc.get('title') or doc.get('filename', 'FastAPI Doc')
    for section in sections:
        fastapi_chunks.append({
            'title': doc_title,
            'filename': doc['filename'],
            'section_content': section
        })

print(f"Created {len(fastapi_chunks)} intelligent section-based chunks.")


Created 1059 intelligent section-based chunks.


In [4]:
# Lexical Index
text_index = Index(
    text_fields=["section_content", "title", "filename"],
    keyword_fields=[]
)
text_index.fit(fastapi_chunks)
print("Lexical index ready.")


Lexical index ready.


In [5]:
# Vector Index
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

embeddings = []
for d in tqdm(fastapi_chunks, desc="Embedding chunks"):
    embeddings.append(embedding_model.encode(d['section_content']))

vector_index = VectorSearch()
vector_index.fit(np.array(embeddings), fastapi_chunks)
print("Vector index ready.")


Embedding chunks: 100%|██████████| 1059/1059 [00:15<00:00, 69.35it/s]

Vector index ready.


In [6]:
def hybrid_search(query: str, num_results: int = 5) -> List[dict]:
    """Combines lexical + semantic search + deduplication"""
    t_results = text_index.search(query, num_results=num_results)
    v_query = embedding_model.encode(query)
    v_results = vector_index.search(v_query, num_results=num_results)
    
    seen = set()
    combined = []
    for res in t_results + v_results:
        uid = res['filename'] + res['section_content'][:80]
        if uid not in seen:
            seen.add(uid)
            combined.append(res)
    return combined[:num_results]


In [7]:
# Custom AsyncOpenAI client for OpenRouter
openai_client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

# Provider + Model 
provider = OpenAIProvider(openai_client=openai_client)

model = OpenAIChatModel(
    "moonshotai/kimi-k2.5",
    provider=provider
)

def search_fastapi_docs(query: str) -> List[dict]:
    """
    Search the FastAPI documentation using hybrid search.
    Returns relevant sections with title, filename and content.
    """
    return hybrid_search(query, num_results=5)

system_prompt = """
You are a FastAPI expert.
Use the search_fastapi_docs tool to find official documentation before answering any technical question.
Always provide accurate code examples (with imports) when they are available.
Always cite sources using this exact format:
[LINK TITLE](https://github.com/fastapi/fastapi/blob/master/FULL_PATH_TO_FILE)
Be clear, concise and helpful.
"""

agent = Agent(
    name="fastapi_agent",
    model=model,
    system_prompt=system_prompt,
    tools=[search_fastapi_docs],
)

print("✅ Pydantic AI agent ready!")


✅ Pydantic AI agent ready!


In [8]:
question = "How do I implement OAuth2 with a password flow in FastAPI?"

result = await agent.run(user_prompt=question)   
print(result.output)   


 Here's how to implement OAuth2 with password flow in FastAPI, using JWT tokens for a production-ready solution:

## Installation

First, install the required packages:

```bash
pip install "fastapi[all]" "python-jose[cryptography]" "passlib[bcrypt]"
```

## Complete Implementation

```python
from datetime import datetime, timedelta, timezone
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from jose import JWTError, jwt
from passlib.context import CryptContext
from pydantic import BaseModel

# to get a string like this run:
# openssl rand -hex 32
SECRET_KEY = "09d25e094faa6ca2556c818166b7a9563b93f7099f6f0f4caa6cf63b88e8d3e7"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30


# --- Pydantic Models ---

class Token(BaseModel):
    access_token: str
    token_type: str


class TokenData(BaseModel):
    username: str | None = None


class User(BaseModel):
    username:

In [9]:
LOG_DIR = Path("fastapi_logs")
LOG_DIR.mkdir(exist_ok=True)

def log_entry(agent, messages, source="user"):
    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": "openrouter",
        "model": "moonshotai/kimi-k2.5",
        "tools": ["search_fastapi_docs"],
        "messages": messages,          
        "source": source
    }

def log_interaction_to_file(agent, result, source="user"):
    messages = json.loads(result.all_messages_json())
    entry = log_entry(agent, messages, source)
    
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)
    filename = f"fastapi_agent_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename
    
    with filepath.open("w", encoding="utf-8") as f:
        json.dump(entry, f, indent=2, default=str)
    
    print(f"Logged → {filepath}")
    return filepath


In [10]:
class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: List[EvaluationCheck]
    summary: str

judge_model = OpenAIChatModel(
    "google/gemma-4-31b-it",
    provider=provider   
)

judge_agent = Agent(
    model=judge_model,
    output_type=EvaluationChecklist
)

eval_prompt = """
Evaluate the FastAPI agent's answer using this checklist.

<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>

Checklist items to check (true/false + short justification):
- instructions_follow
- instructions_avoid
- answer_relevant
- answer_clear
- answer_citations
- completeness
- tool_call_search

Return a structured EvaluationChecklist.
"""


In [11]:
async def run_full_evaluation(num_questions: int = 10):
    sample = random.sample(fastapi_chunks, num_questions)
    prompt_docs = "\n\n".join([c["section_content"][:500] for c in sample])
    test_questions = [f"Question about: {c['title']}" for c in sample]  

    logs = []
    for q in test_questions:
        result = await agent.run(user_prompt=q)
        log_path = log_interaction_to_file(agent, result, source="generated")
        logs.append(log_path)

    print("Running LLM-as-a-Judge...")
    evaluations = []
    def get_text_content(msg):
        for part in msg.get('parts', []):
            if part.get('part_kind') in ['user-prompt', 'text']:
                return part.get('content') or part.get('text')
        return str(msg)

    for log_file in logs:
        with open(log_file, "r", encoding="utf-8") as f:
            log_record = json.load(f)
        
        prompt = eval_prompt.format(
            question=get_text_content(log_record["messages"][-2]),
            answer=get_text_content(log_record["messages"][-1]),
            log=json.dumps(log_record, default=str)
        )
        eval_result = await judge_agent.run(user_prompt=prompt)
        evaluations.append(eval_result)

    # Metrics
    total = len(evaluations)
    metrics = {}
    for check in ["instructions_follow", "instructions_avoid", "answer_relevant",
                  "answer_clear", "answer_citations", "completeness", "tool_call_search"]:
        passed = sum(1 for e in evaluations if any(c.check_name == check and c.check_pass for c in e.output.checklist))
        rate = (passed / total) * 100
        metrics[check] = rate
        print(f"{check:20} : {rate:6.1f}%")

    overall = sum(metrics.values()) / len(metrics)
    print(f"\nOVERALL SCORE : {overall:6.1f}%")
    return evaluations, metrics

# Run it
evals, metrics = await run_full_evaluation(num_questions=10)


Logged → fastapi_logs\fastapi_agent_20260421_195851_7873e4.json
Logged → fastapi_logs\fastapi_agent_20260421_200637_44b215.json
Logged → fastapi_logs\fastapi_agent_20260421_200710_162926.json
Logged → fastapi_logs\fastapi_agent_20260421_200734_26865f.json
Logged → fastapi_logs\fastapi_agent_20260421_200811_14ce39.json
Logged → fastapi_logs\fastapi_agent_20260421_200947_840cdc.json
Logged → fastapi_logs\fastapi_agent_20260421_201059_fb7cd1.json
Logged → fastapi_logs\fastapi_agent_20260421_201159_989047.json
Logged → fastapi_logs\fastapi_agent_20260421_201232_cca0fd.json
Logged → fastapi_logs\fastapi_agent_20260421_201312_444efc.json
Running LLM-as-a-Judge...
instructions_follow  :  100.0%
instructions_avoid   :  100.0%
answer_relevant      :  100.0%
answer_clear         :  100.0%
answer_citations     :  100.0%
completeness         :  100.0%
tool_call_search     :  100.0%

OVERALL SCORE :  100.0%


In [13]:
import pickle
from pathlib import Path

agent_package = {
    "text_index": text_index,
    "vector_index": vector_index,
    "embedding_model": embedding_model,
    "system_prompt": system_prompt,
    "model_name": "moonshotai/kimi-k2.5"
}

Path("agent_package").mkdir(exist_ok=True)
with open("agent_package/fastapi_agent.pkl", "wb") as f:
    pickle.dump(agent_package, f)

print("✅ Agent package saved to 'agent_package/fastapi_agent.pkl'")


✅ Agent package saved to 'agent_package/fastapi_agent.pkl'
